# Field validation — `stratification` (DEPTH pipeline)

| | |
|---|---|
| Subset | `stratification` |
| Pipeline | DEPTH |
| Timestep | 2012-11-09 12:00:00 |
| Domain | one 720 × 720 × 51 tile (≈1400 × 1400 km), set in Section 1 |
| Depth levels | `sfc`, `z25m`, `mld`, `mld_mean` |
| Data | computed on the fly from `s3://dbof/LLC4320_RAW/DEPTH/` |
| Plan | `prompts/field_validation_depth.md` |
| Field reference | `docs/Fields.md` |

Rows of every figure are **depth levels**, not regions — that is the one
structural difference from the surface notebooks.

This is the **template** depth notebook: the shortest chain in the DEPTH pipeline (two raw tracers, one shared density step, three finals), so the machinery is visible rather than buried.  MLD is validated HERE and referenced by every other depth notebook — all the `_mld` and `_mld_mean` channels in the project rest on it.

## Section 1 — Setup

Everything configurable is in the next cell: the **region**, the date,
the depth levels, and the zoom size.  Change `REGION` to validate a
different part of the ocean — any key in `dbof.plotting.regions.REGIONS`
that carries a `zoom` anchor.

Default is the Gulf Stream, anchored at 60°W / 37°N — dynamically
active in every field this project computes, and the same point the
surface notebooks zoom into, so surface and depth look at the same
water.


In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"  # the only DEPTH date transferred so far
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0                  # -> a 200 x 200 km zoom box

SUBSET   = "stratification"
PIPELINE = "DEPTH"
RAW_VARS = ["Theta", "Salt"]
# ------------------------------------------------------------------------

import dask
import numpy as np

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile
from dbof.global_dataset_creation.subset_definitions import (
    get_compute_fn, get_subset_definition, expand_channels_with_suffixes,
)

# tile_utils sets the Agg backend when it is imported (it writes QA PNGs
# on headless nodes), so switch back to inline AFTER the dbof imports or
# no figure in this notebook will render.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

# Channel list straight from the pipeline's own definition -- if the
# subset gains a channel, this notebook picks it up without an edit.
defn = get_subset_definition(PIPELINE, SUBSET)
CHANNELS = expand_channels_with_suffixes(
    defn["compute_features_channels"], list(LEVELS),
    defn.get("extra_channels"),
)
print(f"subset   : {SUBSET}")
print(f"channels : {CHANNELS}")

## Section 2 — Load the tile and compute the fields

We do **not** run `generate-global` here.  That would compute the whole
planet in order to look at one place.

Instead this notebook works on **one tile** — the 720 × 720 × 51 block
the `dbof.tiles` workflow already defines: one LLC face, the full water
column, about 1400 × 1400 km, centred on the region's anchor.  A tile is
*exactly one chunk* of the depth store, so loading it costs one S3 GET
per variable (~106 MB per 3D field).  Tiles are 720-aligned and faces
are 6 × 720 wide, so a tile can never straddle two faces.

Then the **production** compute function for this subset runs on it,
and internally applies the four depth strategies.  Same code as
production, one tile's worth of data.

One thing this costs us: the tile's xgcm grid has **no face
connections**, so cells near the boundary have no neighbours and their
horizontal gradients are wrong.  That rim is NaN'd, using the per-field
widths `tiles/field_registry.py` already records (0 for purely vertical
fields, 1 for staggered interpolation, 3 for gradient and Jacobian
chains).

**A tile samples the region, it does not cover it.**  "Gulf Stream"
here means the ~1400 km tile around 60°W / 37°N — not the whole
80–40°W box the surface notebooks use as a row.


In [ ]:
# Anchor -> rect pixel -> the tile that contains it.
S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)

i_rect, j_rect = tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3)
tile = rect_ij_to_tile(i_rect, j_rect)
print(f"region : {REGION} anchored at ({ANCHOR_LON}, {ANCHOR_LAT})")
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}, "
      f"j={tile.j_face_slice}, i={tile.i_face_slice}")

# Load the tile + its grid, then merge and build a LOCAL xgcm grid.
ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(S3, DATE, tile, RAW_VARS)
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)
print(f"extent : lon [{XC.min():.2f}, {XC.max():.2f}], "
      f"lat [{YC.min():.2f}, {YC.max():.2f}], "
      f"land {100 * LAND.mean():.1f}%")

### The finals, and the intermediates the figures need

`get_compute_fn("DEPTH", SUBSET)` is the production entry point — the
same callable `generate-global` dispatches to — so the finals below are
the pipeline's own numbers.

The **intermediates** are a different matter: the global products never
store them, so they are recomputed here from the same `ds_merge` the
finals came from.  That is deliberate — it means each figure's chain
shows the actual steps, not a reconstruction.


In [ ]:
# Production finals for this subset, on the tile.
store = get_compute_fn(PIPELINE, SUBSET)(ds_merge, xgrid, CHANNELS)
print(f"computed : {sorted(store)}")

# Live intermediates: the raw tracers and the shared density step.
mld = CFAD.mixed_layer_depth(ds_merge)
live = dfig.compute_levels(
    {
        "Theta":     ds_merge["Theta"],
        "Salt":      ds_merge["Salt"],
        "rho_theta": CF.potential_density(ds_merge),
    },
    ds_merge, mld=mld, levels=LEVELS,
)

In [ ]:
# How wide the invalid rim is for this subset, straight from the tile
# registry (0 here: nothing in this chain takes a horizontal gradient).
EDGE_MARGIN = dfig.edge_margin_for(
    list(defn["compute_features_channels"])
    + list(defn.get("extra_channels") or []))

# NaN that rim, mask land with the surface hFacC (what production does),
# and reshape into the {base: {level: (x, y, arr)}} the figures take.
level_arrays = dfig.pack_tile_levels(
    {**live, **store}, XC, YC, edge_margin=EDGE_MARGIN,
    land_mask=LAND, levels=LEVELS)

## Section 3 — Subset: `stratification`

Channels, verbatim from `subset_definitions.DEPTH_SUBSETS`:

| Channel | Kind |
|---|---|
| `N2_sfc`, `N2_z25m`, `N2_mld`, `N2_mld_mean` | base × depth suffixes |
| `mixed_layer_depth` | extra (inherently 2D) |
| `ml_heat_content` | extra (inherently 2D) |

The two extras are 2D by nature, so they repeat down the depth rows of
every figure rather than varying with the row.


## Section 4 — Field & dependency table

| FIELD | UNITS | EQUATION | DEPENDS ON | CODE |
|---|---|---|---|---|
| `rho_theta` | kg m⁻³ | JMD95(S, Θ, p = 0) | Theta, Salt | `calculate_fields.potential_density` |
| `N2_{sfx}` | s⁻² | N² = (g/ρ₀)·∂ρ/∂z | rho_theta, Z, drF | `calculate_fields_at_depth.buoyancy_frequency_squared` |
| `mixed_layer_depth` | m | deepest z with σ₀ − σ₀(10 m) ≤ 0.03 kg m⁻³ | rho_theta, Z | `calculate_fields_at_depth.mixed_layer_depth` |
| `ml_heat_content` | J m⁻² | Q = ∫₀^MLD c_p·ρ₀·Θ dz | Theta, MLD, drF | `calculate_fields_at_depth.mixed_layer_heat_content` |

σ₀ is `rho_theta − 1000`; the constant offset drops out of both the
vertical derivative and the MLD threshold, so the figures plot
`rho_theta` and the two are interchangeable here.

**Processing operations in play:** depth selection and averaging
(`depth_strategies`: k = 0 / nearest-k to 25 m / nearest-k to MLD /
thickness-weighted mean over the mixed layer), the vertical derivative
on the tracer grid (`vertical_helpers`, float64 internally), land
masking from the surface `hFacC`, and the tile edge rim.

**Tile edge rim:** every field here is purely vertical — no horizontal
stencil — so `field_registry` gives them `edge_margin = 0` and nothing
is lost at the tile boundary.  That changes from `frontal_structure`
onwards, where the gradient chains carry a 3-cell rim.

**Gradient artifacts:** none of these fields takes a horizontal
gradient, so the sparkle cases in `docs/Gradients.md` do not arise in
this notebook.  They start at `frontal_structure` and `kinematic`.

**Depth clipping:** the store keeps only the top 51 levels (≈969 m).
Any mixed layer deeper than that is clipped — visible in winter
deep-convection regions, not in the Gulf Stream in November.


## Section 5 — Per-field validation

Two figures per field.

**Figure 1 — maps.**  Columns are the dependency chain, raw → final.
Rows are the four depth levels over the whole tile, then the same four
zoomed to a 200 × 200 km box; the crimson square on the whole-tile rows
is where the zoom is.  One colour scale per column, shared by every row
including the zooms, so nothing changes colour when you look closer.

**Figure 2 — PDFs.**  Four rows, the whole tile at each level.  Bins
are shared down a column, so reading a column top to bottom shows how
the distribution changes with depth.  The zoom boxes are deliberately
absent — too few cells to make an honest histogram.

Fields that are inherently 2D (`Eta`, `gradeta2`, `ug`, `vg`,
`coriolis_f`, `mixed_layer_depth`, `ml_heat_content`) repeat down the
rows; `pack_tile_levels` printed which ones did.

("Grid" in the function names below means the rows × columns array of
panels — not the model's Arakawa C-grid, which is `docs/Grid.md`.)


In [ ]:
# Section 5 helpers: one call per figure, shared by every field.
CHAINS = {
    "N2": ["Theta", "Salt", "rho_theta", "N2"],
    "mixed_layer_depth": ["Theta", "Salt", "rho_theta", "mixed_layer_depth"],
    "ml_heat_content": ["Theta", "mixed_layer_depth", "ml_heat_content"],
}
LOG_FIELDS = set()


def figure1_maps(field):
    """Figure 1 for one field: chain across, depth down."""
    dfig.depth_map_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        region=REGION, levels=LEVELS,
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        zoom_half_km=ZOOM_HALF_KM,
        suptitle=(f"Figure 1 — {field} | {REGION} tile | "
                  f"columns = dependency chain, rows = depth "
                  f"(lower 4 rows: {2 * ZOOM_HALF_KM:.0f} km zoom)"),
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2 for one field: PDFs, chain across, depth down."""
    dfig.depth_pdf_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        levels=LEVELS, log10_fields=LOG_FIELDS,
        suptitle=(f"Figure 2 — {field} | {REGION} tile | density; "
                  f"land + rim NaNs dropped; bins shared down each column"),
    )
    plt.show()

### N² — buoyancy frequency squared

Stratification strength.  Expect a clear ordering with depth: largest just below the surface where the seasonal thermocline sits, near-zero through the mixed layer, and the `_mld_mean` row smoother than the `_mld` row because it averages over the layer instead of sampling one level.  Negative values are real (statically unstable cells) and should be sparse and speckled, not organised.

In [ ]:
figure1_maps("N2")

In [ ]:
figure2_pdfs("N2")

### MLD — mixed layer depth

The 0.03 kg m⁻³ threshold MLD.  This is the field every other depth notebook leans on, so it gets validated here and referenced elsewhere.  On the Gulf Stream tile expect deeper values north of the front and shallow values in the warm core; sharp lateral steps across the front are physical.  The four rows are identical by construction — MLD has no depth dependence.

In [ ]:
figure1_maps("mixed_layer_depth")

In [ ]:
figure2_pdfs("mixed_layer_depth")

### Q_ml — mixed-layer heat content

∫ c_p·ρ₀·Θ over the mixed layer, so it inherits both the MLD pattern and the temperature pattern: large where the layer is both deep and warm.  Also depth-independent, so the rows repeat.

In [ ]:
figure1_maps("ml_heat_content")

In [ ]:
figure2_pdfs("ml_heat_content")

## Section 6 — Literature comparison

**PENDING — nothing to build here yet.**

The comparison figure is chosen *after* the literature figure is, not
before.  Once LH picks a paper figure and drops the PNG into
`../literature_figures/` (naming convention
`{field(s)}_{Citation}_{description}.png`), we decide which of our
panels belongs beside it and add a subsection here — one subsection per
reference, using `dbof.plotting.literature_comparison.side_by_side`.

Leave this section as-is until then.


## Summary — did every channel come out sane?

Coverage and range for each channel at each level, then the physical
checks that are worth failing loudly on.


In [ ]:
# Coverage + range per field per level.
print(f"{'field':<22}{'level':<10}{'finite %':>9}"
      f"{'min':>14}{'max':>14}")
print("-" * 69)
for field in sorted(level_arrays):
    for lev in LEVELS:
        arr = level_arrays[field][lev][2]
        finite = np.isfinite(arr)
        pct = 100.0 * finite.mean()
        lo = np.nanmin(arr) if finite.any() else np.nan
        hi = np.nanmax(arr) if finite.any() else np.nan
        print(f"{field:<22}{lev:<10}{pct:>8.1f}%{lo:>14.4g}{hi:>14.4g}")

In [ ]:
# Physical checks.  These assert -- a red cell here is a real problem.
mld_arr = level_arrays["mixed_layer_depth"]["sfc"][2]
n2_sfc = level_arrays["N2"]["sfc"][2]
n2_mld = level_arrays["N2"]["mld"][2]
q_ml = level_arrays["ml_heat_content"]["sfc"][2]

CHECKS = [
    ("MLD positive",
     np.nanmin(mld_arr) > 0,
     f"min = {np.nanmin(mld_arr):.1f} m"),
    ("MLD within the stored water column (<= 969 m)",
     np.nanmax(mld_arr) <= 969.0,
     f"max = {np.nanmax(mld_arr):.1f} m"),
    ("N2 mostly stable (>0) at the surface",
     np.nanmean(n2_sfc > 0) > 0.9,
     f"{100 * np.nanmean(n2_sfc > 0):.1f}% positive"),
    ("N2 weaker inside the mixed layer than at the surface",
     np.nanmedian(np.abs(n2_mld)) < np.nanmedian(np.abs(n2_sfc)),
     f"median |N2| mld = {np.nanmedian(np.abs(n2_mld)):.2e}, "
     f"sfc = {np.nanmedian(np.abs(n2_sfc)):.2e}"),
    ("ML heat content positive",
     np.nanmin(q_ml) > 0,
     f"min = {np.nanmin(q_ml):.3e} J m-2"),
    ("tile is not mostly land",
     np.isfinite(mld_arr).mean() > 0.5,
     f"{100 * np.isfinite(mld_arr).mean():.1f}% finite"),
]
failures = []
for name, ok, detail in CHECKS:
    print(f"{'OK  ' if ok else 'FAIL'}  {name}  ({detail})")
    if not ok:
        failures.append(name)
assert not failures, f"physical checks failed: {failures}"
print("\nAll physical checks passed.")

---

### Cross-references

- **MLD** is validated here and nowhere else.  Every notebook with a
  `_mld` / `_mld_mean` channel, plus `Fr` and `KE`, points back to this
  section rather than re-deriving it.
- **σ₀ / buoyancy** as output channels live in
  `surface_fields/frontal_structure.ipynb`.
- **The tile workflow** — what a tile is, how the rect index maps to a
  face, the `edge_margin` convention — `docs/Tiles.md` and
  `src/dbof/tiles/`.
- **Gradient / interpolation artifacts** — `docs/Gradients.md`; the
  evidence notebook is `../field_validation_sparkle.ipynb`.  Nothing in
  this subset takes a horizontal gradient, so none of it applies here.
- **The model grid** (C-grid staggering, the 51 stored levels and their
  real depths) — `../Grid.ipynb` and `docs/Grid.md`.
